# 06 - 配置驱动与 Transform

> **何时使用**: 当你需要复用配置、批量生成多表数据、或在生成后对数据进行变换时。
>
> **核心概念**: Pydantic 配置模型 + YAML/JSON 文件 + Transform 脚本 + Snapshot 快照。

## 适用场景

- 多表批量生成 → YAML 配置 + `fill_from_config()`
- 复杂业务逻辑（如条件计算）→ Transform Script
- CI/CD 可复现测试数据 → Snapshot + `replay()`
- 团队共享测试环境 → YAML 配置提交到 Git

## 你将学到

- 配置模型层次：GeneratorConfig → TableConfig → ColumnConfig
- YAML/JSON 配置格式
- Transform Scripts 业务逻辑
- ColumnAssociation 跨表关联
- SnapshotManager 快照管理

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| **→ 06** | **配置驱动与 Transform** | **Config / Core** | **01** |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
ORG_QUERY = "SELECT org_code FROM organizations"
ORG_PATTERN = "ORG-\\d{4}"
from sqlseed.config.models import GeneratorConfig, TableConfig, ColumnConfig, ProviderType, ColumnConstraintsConfig, ColumnAssociation
from sqlseed.config.loader import save_config, load_config, generate_template
from sqlseed.config.snapshot import SnapshotManager
from pathlib import Path

import sqlite3
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| Transform 加载 | `src/sqlseed/core/transform.py` | `TransformLoader` |
| 配置模型 | `src/sqlseed/config/models.py` | `GeneratorConfig` |

> 对应架构图: [§9 配置模型层次结构](../docs/architecture.zh-CN.md#9-配置模型层次结构)

## 1. 先看效果 — YAML 配置驱动批量填充

对于复杂的多表场景，用 YAML 配置文件声明所有表和列，一行代码批量填充：

```yaml
db_path: "app.db"
tables:
  - name: users
    count: 10000
    columns:
      - name: email
        generator: email
```

In [2]:

# 用 Python 对象构建配置（等价于 YAML）
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    tables=[
        TableConfig(name="organizations", count=5, clear_before=True),
        TableConfig(name="members", count=10, clear_before=True),
    ]
)
config_path = Path("_demo_config.yaml")
save_config(config, str(config_path))

# 一行代码，批量填充多表
results = fill_from_config(str(config_path))
print(f"{'表名':<15s}  {'行数':>6s}  {'耗时':>8s}  {'速度':>10s}")
print('-' * 45)
for r in results:
    print(f"{r.table_name:<15s}  {r.count:>6d}  {r.elapsed:>7.3f}s  {r.rows_per_second:>8.0f} rows/s")

config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

表名                   行数        耗时          速度
---------------------------------------------
organizations         5    0.031s       163 rows/s
members              10    0.033s       299 rows/s


配置驱动的好处：

- **可版本控制** — YAML 文件可提交到 Git
- **可复现** — 相同配置 = 相同数据（配合 seed）
- **可共享** — 团队成员用同一份配置

下面详细拆解配置模型的每个层次。

## 2. 配置模型层次

sqlseed 使用 Pydantic 模型定义配置层次：

```
GeneratorConfig
├── db_path: str
├── provider: ProviderType (MIMESIS)
├── locale: str (en_US)
├── tables: list[TableConfig]
│   └── TableConfig
│       ├── name: str
│       ├── count: int (1000)
│       ├── columns: list[ColumnConfig]
│       │   └── ColumnConfig
│       │       ├── Source模式: generator + params
│       │       └── Derived模式: derive_from + expression
│       └── clear_before, seed, transform, enrich
├── associations: list[ColumnAssociation]
└── optimize_pragma, log_level, snapshot_dir
```

In [3]:
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    locale="en_US",
    tables=[
        TableConfig(
            name="organizations",
            count=5,
            columns=[
                ColumnConfig(name="name", generator="company"),
                ColumnConfig(name="description", generator="sentence"),
            ],
            clear_before=True,
        ),
    ],
)
print(f"GeneratorConfig: db_path={Path(config.db_path).name}")
print(f"  provider={config.provider}, locale={config.locale}")
print(f"  tables: {[t.name for t in config.tables]}")
print(f"  TableConfig[0]: count={config.tables[0].count}, columns={len(config.tables[0].columns)}")

GeneratorConfig: db_path=sqlseed_demo.db
  provider=ProviderType.MIMESIS, locale=en_US
  tables: ['organizations']
  TableConfig[0]: count=5, columns=2


## 3. ColumnConfig 双模式验证

ColumnConfig 有两种互斥模式：
- **Source 模式**：`generator` + `params` + `null_ratio` + `provider`
- **Derived 模式**：`derive_from` + `expression`

Pydantic `model_validator` 强制互斥，同时设置 `generator` 和 `derive_from` 会报错。

In [4]:
try:
    bad_config = ColumnConfig(
        name="test",
        generator="email",
        derive_from="other_col",
    )
except Exception as e:
    print(f"❌ 双模式冲突: {type(e).__name__}")
    print(f"   {e}")

print("\n✅ Source 模式 (generator):")
src = ColumnConfig(name="email_col", generator="email")
print(f"   generator={src.generator}, derive_from={src.derive_from}")

print("\n✅ Derived 模式 (derive_from):")
drv = ColumnConfig(name="short_code", derive_from="project_no", expression="value[-6:]")
print(f"   generator={drv.generator}, derive_from={drv.derive_from}, expression={drv.expression}")

❌ 双模式冲突: ValidationError
   1 validation error for ColumnConfig
  Value error, Column 'test': cannot use both 'generator' and 'derive_from' [type=value_error, input_value={'name': 'test', 'generat...rive_from': 'other_col'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

✅ Source 模式 (generator):
   generator=email, derive_from=None

✅ Derived 模式 (derive_from):
   generator=None, derive_from=project_no, expression=value[-6:]


## 4. 完整 YAML 配置实战

In [5]:
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    tables=[
        TableConfig(name="organizations", count=5, clear_before=True),
        TableConfig(
            name="members",
            count=20,
            clear_before=True,
            columns=[
                ColumnConfig(name="member_no", generator="pattern", params={"pattern": "M-\\d{6}"}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
                ColumnConfig(name="email", generator="email", constraints=ColumnConstraintsConfig(unique=True)),
            ],
        ),
        TableConfig(
            name="projects",
            count=10,
            clear_before=True,
            columns=[
                ColumnConfig(name="project_no", generator="pattern", params={"pattern": "PRJ-\\d{6}"}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
            ],
        ),
    ],
)
config_path = Path("_demo_config.yaml")
save_config(config, str(config_path))

results = fill_from_config(str(config_path))
for r in results:
    status = "✅" if r.count > 0 else "❌"
    print(f"{status} {r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

✅ organizations: 5 rows in 0.033s
✅ members: 20 rows in 0.059s
✅ projects: 10 rows in 0.043s


## 5. JSON 配置格式

`save_config` 和 `load_config` 同时支持 YAML 和 JSON 格式，通过文件扩展名自动判断。

In [6]:
import json

json_path = Path("_demo_config.json")

save_config(config, str(json_path))
with open(json_path) as f:
    data = json.load(f)
print("JSON 配置格式:")
safe = json.dumps(data, indent=2, ensure_ascii=False)
print(safe[:500] + "...")

json_path.unlink(missing_ok=True)

JSON 配置格式:
{
  "db_path": "/Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db",
  "provider": "mimesis",
  "locale": "en_US",
  "tables": [
    {
      "name": "organizations",
      "count": 5,
      "batch_size": 5000,
      "columns": [],
      "clear_before": true,
      "seed": null,
      "transform": null,
      "enrich": false
    },
    {
      "name": "members",
      "count": 20,
      "batch_size": 5000,
      "columns": [
        {
          "name": "member_no",
          "genera...


## 6. load_config 演示

In [7]:
loaded = load_config(str(config_path))
print("load_config 结果:")
print(f"  db_path: {Path(loaded.db_path).name}")
print(f"  provider: {loaded.provider}")
print(f"  tables: {[t.name for t in loaded.tables]}")
print(f"  tables[1].columns: {[c.name for c in loaded.tables[1].columns]}")

load_config 结果:
  db_path: sqlseed_demo.db
  provider: ProviderType.MIMESIS
  tables: ['organizations', 'members', 'projects']
  tables[1].columns: ['member_no', 'email']


## 7. generate_template / sqlseed init

`generate_template()` 根据 DB schema 自动生成配置模板，CLI 对应 `sqlseed init` 命令。

In [8]:
template = generate_template(str(db_path), table_name="organizations")
print("generate_template 结果:")
print(f"  db_path: {Path(template.db_path).name}")
print(f"  tables: {[t.name for t in template.tables]}")
if template.tables:
    t = template.tables[0]
    print(f"  '{t.name}' columns ({len(t.columns)}):")
    for c in t.columns[:5]:
        print(f"    - {c.name}: generator={c.generator}")

generate_template 结果:
  db_path: sqlseed_demo.db
  tables: ['organizations']
  'organizations' columns (0):


## 8. Transform Scripts — 复杂业务逻辑

当声明式配置无法表达的业务逻辑，可以用 Python Transform Script 实现。脚本需定义 `transform_row(row, ctx)` 函数，在数据生成后、写入数据库前对每行进行变换。

**典型场景**：
- 条件计算（如根据年龄算 VIP 等级）
- 数据格式化（如手机号加国际区号）
- 字段组合（如拼接 full_name）
- 数据校验与修正

In [9]:
transform_script = Path("_demo_transform.py")
transform_script.write_text(
    "def transform_row(row, ctx):\n"
    "    # 根据 member_count 计算组织规模等级\n"
    "    count = row.get('member_count', 0) or 0\n"
    "    if count >= 200:\n"
    '        row[\'description\'] = f"[大型组织] {row.get(\'name\', \'\')} - 全球领先的技术企业"\n'
    "    elif count >= 50:\n"
    '        row[\'description\'] = f"[中型组织] {row.get(\'name\', \'\')} - 快速成长的科技公司"\n'
    "    else:\n"
    '        row[\'description\'] = f"[小型组织] {row.get(\'name\', \'\')} - 创新型初创企业"\n'
    "    # 名称转大写\n"
    "    if row.get('name'):\n"
    "        row['name'] = row['name'].upper()\n"
    "    return row\n"
)

with connect(str(db_path)) as orch:
    result = orch.fill_table("organizations", count=5, clear_before=True, transform=str(transform_script),
        columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                 "parent_code": {"type": "choice", "choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute(ORG_QUERY).fetchall()]]}})
    print(f"Transform 填充: {result.count} rows")

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT name, member_count, description FROM organizations").fetchall()
for r in rows:
    desc = str(r[2])[:60]
    print(f"  {r[0]:<20s} | members={r[1]:4d} | {desc}")
conn.close()

transform_script.unlink(missing_ok=True)


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Transform 填充: 5 rows
  DOUGLASS HOUSE       | members=   0 | [小型组织] Douglass House - 创新型初创企业
  SONG OLSEN           | members=   0 | [小型组织] Song Olsen - 创新型初创企业
  ANTONINA TERRY       | members=   0 | [小型组织] Antonina Terry - 创新型初创企业
  VI HINES             | members=   0 | [小型组织] Vi Hines - 创新型初创企业
  LYNDIA SAUNDERS      | members=   0 | [小型组织] Lyndia Saunders - 创新型初创企业


## 9. ColumnAssociation 跨表关联

`ColumnAssociation` 声明跨表共享同一列值，确保 FK 引用一致性。

In [10]:
config_with_assoc = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name="organizations", count=3, clear_before=True, columns=[
            ColumnConfig(name="org_code", generator="pattern", params={"regex": ORG_PATTERN}),
            ColumnConfig(name="parent_code", generator="choice", params={"choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute(ORG_QUERY).fetchall()]]}),
        ]),
        TableConfig(name="members", count=10, clear_before=True),
    ],
    associations=[
        ColumnAssociation(
            column_name="org_code",
            source_table="organizations",
            target_tables=["members"],
            strategy="shared_pool",
        ),
    ],
)
assoc_path = Path("_assoc_config.yaml")
save_config(config_with_assoc, str(assoc_path))

results = fill_from_config(str(assoc_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows")

conn = sqlite3.connect(str(db_path))
org_codes = [r[0] for r in conn.execute("SELECT DISTINCT org_code FROM organizations").fetchall()]
member_orgs = [r[0] for r in conn.execute("SELECT org_code FROM members LIMIT 5").fetchall()]
print(f"\n关联验证: org_codes={org_codes}")
print(f"  members 的 org_code: {member_orgs}")
all_valid = all(m in org_codes for m in member_orgs if m)
print(f"  所有 FK 引用有效: {all_valid}")
conn.close()

assoc_path.unlink(missing_ok=True)


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

  organizations: 3 rows
  members: 10 rows

关联验证: org_codes=['ORG-2543', 'ORG-2948', 'ORG-8473']
  members 的 org_code: ['ORG-2948', 'ORG-8473', 'ORG-8473', 'ORG-8473', 'ORG-8473']
  所有 FK 引用有效: True


## 10. ColumnConstraintsConfig 约束配置

`ColumnConstraintsConfig` 支持以下约束：
- `unique`: 唯一约束
- `min_value` / `max_value`: 数值范围
- `regex`: 正则匹配
- `max_retries`: 最大重试次数

In [11]:
constrained = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(
            name="organizations",
            count=5,
            clear_before=True,
            columns=[
                ColumnConfig(name="name", generator="company", constraints=ColumnConstraintsConfig(unique=True)),
                ColumnConfig(name="org_code", generator="pattern", params={"pattern": ORG_PATTERN}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
            ],
        ),
    ],
)
c_path = Path("_constraints_config.yaml")
save_config(constrained, str(c_path))

results = fill_from_config(str(c_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows")

conn = sqlite3.connect(str(db_path))
names = [r[0] for r in conn.execute("SELECT name FROM organizations").fetchall()]
codes = [r[0] for r in conn.execute(ORG_QUERY).fetchall()]
print(f"\nUNIQUE name: {len(names) == len(set(names))}")
print(f"UNIQUE org_code: {len(codes) == len(set(codes))}")
conn.close()

c_path.unlink(missing_ok=True)
config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

  organizations: 5 rows

UNIQUE name: True
UNIQUE org_code: True


## 11. SnapshotManager 快照管理

SnapshotManager 支持保存、加载、列出和回放数据快照，CLI 对应 `sqlseed replay` 命令。

In [12]:

snap_mgr = SnapshotManager()  # Uses platform cache dir (~/Library/Caches/sqlseed/snapshots on macOS)

config = GeneratorConfig(
    db_path=str(db_path),
    tables=[TableConfig(name="organizations", count=3, clear_before=True)],
)
snapshot_path = snap_mgr.save(config, "organizations", count=3, seed=42)
print(f"快照保存: {Path(snapshot_path).name}")
print(f"快照目录: {snap_mgr._snapshot_dir}")

snapshots = snap_mgr.list_snapshots()
print(f"已有快照数: {len(snapshots)}")

data = snap_mgr.load(snapshot_path)
print(f"快照内容: table={data.get('table_name')}, count={data.get('count')}")

# 回放快照 — 完全复现之前的生成
result = snap_mgr.replay(snapshot_path)
print(f"\n快照回放: {result}")

快照保存: 2026-05-06_074253_organizations.yaml
快照目录: /Users/sunbo/Library/Caches/sqlseed/snapshots
已有快照数: 77
快照内容: table=organizations, count=3


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]


快照回放: GenerationResult(table=organizations, count=3, elapsed=0.04s, speed=85.51 rows/s)


### CLI 快照命令

```bash
# 生成并保存快照
sqlseed fill app.db --table users --count 10000 --seed 42 --snapshot
# → Snapshot saved: snapshots/2026-04-15_033000_users.yaml

# 回放快照
sqlseed replay snapshots/2026-04-15_033000_users.yaml
```

**典型场景**：CI/CD 可复现测试数据、团队共享一致的测试环境、快速重建数据库状态。

## 12. Preview & Debug CLI

`sqlseed preview` 预览数据不写入，`sqlseed inspect --show-mapping` 查看列映射策略。

In [13]:
from click.testing import CliRunner

from sqlseed.cli.main import cli

preview_rows = preview(str(db_path), table="organizations", count=3,
                       columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                                "name": {"type": "company"}})
print("preview 结果:")
for row in preview_rows:
    print(f"  {row.get('org_code', 'N/A')} | {row.get('name', 'N/A')}")



runner = CliRunner()
result = runner.invoke(cli, ["inspect", str(db_path), "-t", "organizations", "--show-mapping"])
if result.output.strip():
    print("\nsqlseed inspect --show-mapping:")
    print(result.output[:500])

                                           Table: organizations (3 rows)                                           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column       ┃ Type        ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                        ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ org_code     │ VARCHAR(16) │ ✗        │ ✓  │      │ string      │ {'min_length': 6, 'max_length': 12,           │
│              │             │          │    │      │             │ 'charset': 'alphanumeric'}                    │
│ name         │ VARCHAR(64) │ ✗        │    │      │ name        │ {}                                            │
│ parent_code  │ VARCHAR(16) │ ✓        │    │      │ foreign_key │ {'ref_table': 'organizations', 'ref_column':  │
│              │             │          │    │      │             │ 'org_code', 'strategy': 'random',             │
│              │             │          │    │      │             │ '_ref_values': ['JmTPSI', 'fLBcbfnoGM',       │
│              │             │          │    │      │             │ 'hbVrpoiVgRV']}                               │
│ description  │ TEXT        │ ✓        │    │      │ text        │ {'min_length': 100, 'max_length': 500}        │
│ is_active    │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ member_count │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ created_at   │ TEXT        │ ✓        │    │      │ datetime    │ {}                                            │
└──────────────┴─────────────┴──────────┴────┴──────┴─────────────┴───────────────────────────────────────────────┘

        Foreign Keys: organizations         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column      ┃ Ref Table     ┃ Ref Column ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ parent_code │ organizations │ org_code   │
└─────────────┴───────────────┴────────────┘

preview 结果:
  ORG-9977 | Enterasys Networks
  ORG-6810 | American Broadcasting Company
  ORG-5959 | Alienware


## ✅ 总结

| 功能 | API | 状态 |
|---|---|---|
| 配置模型层次 | GeneratorConfig/TableConfig/ColumnConfig | ✅ |
| 双模式验证 | ColumnConfig model_validator | ✅ |
| YAML/JSON 配置 | save_config/load_config | ✅ |
| 自动生成模板 | generate_template / sqlseed init | ✅ |
| Transform Scripts | transform_row(row, ctx) | ✅ |
| 跨表关联 | ColumnAssociation | ✅ |
| 约束配置 | ColumnConstraintsConfig | ✅ |
| 快照管理 | SnapshotManager | ✅ |
| 预览与调试 | preview / inspect --show-mapping | ✅ |

**下一步**: [07-ai-plugin.ipynb](07-ai-plugin.ipynb) — AI 智能配置

In [14]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
